# LFD-Net - Remote Sensing Image Dehazing

This notebook tests pretrained weights and trains LFD-Net on the SateHaze1k-thick dataset.

**Dataset**: SateHaze1k-thick  
**Model**: LFD-Net (Light Feature Dehaze Net)  
**Metrics**: PSNR, SSIM, SAM, ERGAS (validation) + comprehensive evaluation

In [ ]:
#@title 1. Install Dependencies
!pip install pytorch-msssim lpips scikit-image thop opencv-python

In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/LFDNet_Results"
!mkdir -p "/content/drive/MyDrive/LFDNet_Results/pretrained_results"
!mkdir -p "/content/drive/MyDrive/LFDNet_Results/trained_results"
!mkdir -p "/content/drive/MyDrive/LFDNet_Results/weights"

In [ ]:
#@title 3. Download SateHaze1k Dataset
!wget -O Haze1k.zip "https://www.dropbox.com/s/k2i3p7puuwl2g59/Haze1k.zip?dl=1"
!unzip -q Haze1k.zip -d ./Haze1k_dataset

!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/target/265.png
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/input/265.png
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/target/271.png
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/input/271.png

!ls -l ./Haze1k_dataset/Haze1k/Haze1k_thick/dataset/

In [ ]:
#@title 4. Clone LFD-Net Repository
!git clone https://github.com/sumire25/LFD-Net.git
%cd LFD-Net
!ls

In [ ]:
#@title 5. Compute FLOPs and Parameters
import torch
from thop import profile
import model

lfd_net = model.LFD_Net().cuda()
lfd_net.eval()

dummy_input = torch.randn(1, 3, 480, 640).cuda()
macs, params = profile(lfd_net, inputs=(dummy_input,))

flops_g = macs / 1e9
params_m = params / 1e6

print(f"\n--- LFD-Net Complexity ---")
print(f"FLOPs: {flops_g:.4f} G")
print(f"Parameters: {params_m:.4f} M")

## Phase 1: Test Pretrained Weights

In [ ]:
#@title 6. Run Inference with Pretrained Weights
!python infer_multi.py \
  -td /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/input/ \
  -sd /content/drive/MyDrive/LFDNet_Results/pretrained_results/ \
  -w pretrained_weights/SateHaze1k-TK_best.pth

In [ ]:
#@title 7. Evaluate Pretrained Model
!python evaluate.py \
  --train_folder /content/drive/MyDrive/LFDNet_Results/pretrained_results/ \
  --target_folder /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/target/

## Phase 2: Train from Scratch

In [ ]:
#@title 8. Create Required Directories
!mkdir -p trained_weights
!mkdir -p training_data_captures

In [ ]:
#@title 9. Train LFD-Net
!python train.py \
  -th /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/train/input/ \
  -to /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/train/target/ \
  -vh /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/valid/input/ \
  -vo /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/valid/target/ \
  -e 50 \
  -lr 0.001 \
  --val_freq 3 \
  --save_dir /content/drive/MyDrive/LFDNet_Results/weights/

## Phase 3: Test Trained Weights

In [ ]:
#@title 10. Run Inference with Trained Weights
!python infer_multi.py \
  -td /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/input/ \
  -sd /content/drive/MyDrive/LFDNet_Results/trained_results/ \
  -w /content/drive/MyDrive/LFDNet_Results/weights/best.pth

In [ ]:
#@title 11. Evaluate Trained Model
!python evaluate.py \
  --train_folder /content/drive/MyDrive/LFDNet_Results/trained_results/ \
  --target_folder /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/target/